# Drive-Thru Voice AI — Fine-Tune (Phase B / §5) — self-checking version

Every step prints **`✅ STEP N OK`** when it worked, or a **`❌`** message telling you what to fix. Run cells **top to bottom**. Don't move on until you see the ✅ for the current step.

### If it says "Disconnected" / you lost the runtime
1. Top-right → **Reconnect** (or **Runtime → Reconnect**).
2. **Runtime → Change runtime type → T4 GPU** → Save.
3. Re-run from **Step 0** down. The upload step (Step 2) **skips if the file is still there**, so re-running is cheap. If the runtime fully reset, you'll just re-upload `dataset.jsonl` once — takes 2 seconds.

Keep this browser tab visible while it runs — free Colab disconnects idle tabs.

## Step 0 — Verify a GPU is attached (do this FIRST)

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout or r.stderr)
assert r.returncode == 0, ('\u274c No GPU. Fix: Runtime > Change runtime type > T4 GPU, '
                           'Save, Reconnect, then re-run this cell.')
print('\u2705 STEP 0 OK \u2014 GPU is attached.')

## Step 1 — Install Unsloth (run both cells)

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-cache-dir --no-deps unsloth unsloth_zoo

In [ ]:
import torch, unsloth
print('unsloth', getattr(unsloth, '__version__', '?'), '| torch', torch.__version__,
      '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), '\u274c PyTorch cannot see the GPU. Re-check Step 0.'
print('\u2705 STEP 1 OK \u2014 Unsloth installed and GPU is visible to PyTorch.')

## Step 2 — Upload your data
Run the cell. If a **`Choose Files`** button appears, click it → in Finder press **⌘⇧G**, paste `/Users/andrewphan/drivethru-voice-ai/data`, pick **`dataset.jsonl`** → Open. (If the file is already uploaded, the cell just confirms it — no button.)

In [ ]:
import os, json
if not os.path.exists('dataset.jsonl'):
    from google.colab import files
    print('Click "Choose Files" below and pick dataset.jsonl from drivethru-voice-ai/data/')
    files.upload()
assert os.path.exists('dataset.jsonl'), '\u274c dataset.jsonl not found \u2014 re-run and upload it.'
rows = [json.loads(l) for l in open('dataset.jsonl') if l.strip()]
assert rows and all('messages' in r for r in rows), '\u274c wrong file \u2014 expected lines with a "messages" list.'
print(f'\u2705 STEP 2 OK \u2014 {len(rows)} dialogues loaded. First one roles:',
      [m['role'] for m in rows[0]['messages']])

## Step 3 — Load Qwen2.5-1.5B in 4-bit + attach LoRA (D2)

In [ ]:
from unsloth import FastLanguageModel
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-1.5B-Instruct', max_seq_length=max_seq_length, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16, lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth', random_state=7)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\u2705 STEP 3 OK \u2014 model loaded; trainable LoRA params: {trainable:,}')

## Step 4 — Format with Qwen's chat template + length check

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset
tokenizer = get_chat_template(tokenizer, chat_template='qwen-2.5')
def fmt(ex):
    return {'text': tokenizer.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False)}
train_ds = load_dataset('json', data_files='dataset.jsonl', split='train').map(fmt)
SYSTEM = train_ds[0]['messages'][0]['content']
lens = [len(tokenizer(t)['input_ids']) for t in train_ds['text']]
print('token lengths  min/median/max =', min(lens), sorted(lens)[len(lens)//2], max(lens))
assert max(lens) <= max_seq_length, f'\u274c an example is {max(lens)} tokens > max_seq_length={max_seq_length}; raise it in Step 3.'
print(train_ds[0]['text'][:500])
print(f'\n\u2705 STEP 4 OK \u2014 {len(train_ds)} examples formatted, all fit in {max_seq_length} tokens.')

## Step 5 — Build trainer (loss on assistant turns only) + verify masking

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train_ds,
    args=SFTConfig(
        dataset_text_field='text', max_seq_length=max_seq_length,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=5, num_train_epochs=3, learning_rate=2e-4,
        logging_steps=1, optim='adamw_8bit', weight_decay=0.01,
        lr_scheduler_type='linear', seed=7, output_dir='outputs', report_to='none'))
trainer = train_on_responses_only(
    trainer, instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
try:
    labels = trainer.train_dataset[0]['labels']
    masked = sum(1 for x in labels if x == -100)
    print(f'masking: {masked}/{len(labels)} tokens masked (system+user hidden from loss)')
    assert 0 < masked < len(labels), '\u274c masking looks wrong \u2014 check the part strings.'
except (KeyError, TypeError):
    print('(masking applied lazily by the collator \u2014 cannot preview, this is fine)')
print('\u2705 STEP 5 OK \u2014 trainer ready.')

## Step 6 — Train (a few minutes; watch the loss tick down)

In [ ]:
stats = trainer.train()
print(f'\n\u2705 STEP 6 OK \u2014 training done. final loss: {stats.training_loss:.3f}')

## Step 7 — Smoke test (does it emit <say> + <order>?)

In [ ]:
FastLanguageModel.for_inference(model)
msgs = [{'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': 'lemme get a double stack combo medium and a large fries'}]
inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
out = model.generate(input_ids=inputs, max_new_tokens=160, use_cache=True, temperature=0.3, do_sample=True)
reply = tokenizer.batch_decode(out)[0].split('<|im_start|>assistant')[-1]
print(reply)
print('\n\u2705 STEP 7 OK \u2014 look for a <say>...</say> and an <order>{...}</order> above.')
print('   (Rough with only the seed \u2014 you are checking the SHAPE, not accuracy.)')

## Step 8 — Export GGUF INT4 (what Ollama runs) (D3)

In [ ]:
model.save_pretrained_gguf('drivethru', tokenizer, quantization_method='q4_k_m')
import glob, os
ggufs = glob.glob('drivethru/*.gguf')
assert ggufs, '\u274c no .gguf produced \u2014 check the cell output above for errors.'
print(f'\u2705 STEP 8 OK \u2014 {ggufs[0]} ({os.path.getsize(ggufs[0])/1e6:.0f} MB)')

## Step 9 — Build Modelfile + download the model to your Mac

In [ ]:
import glob, os
gguf = glob.glob('drivethru/*.gguf')[0]
modelfile = (
    f'FROM ./{os.path.basename(gguf)}\n'
    'PARAMETER temperature 0.3\n'
    'PARAMETER stop "<|im_end|>"\n'
    'TEMPLATE """{{ if .System }}<|im_start|>system\n{{ .System }}<|im_end|>\n{{ end }}'
    '{{ if .Prompt }}<|im_start|>user\n{{ .Prompt }}<|im_end|>\n{{ end }}'
    '<|im_start|>assistant\n{{ .Response }}<|im_end|>\n"""\n'
    'SYSTEM """' + SYSTEM + '"""\n')
open('drivethru/Modelfile', 'w').write(modelfile)
!cd drivethru && zip -r ../drivethru_model.zip . >/dev/null
assert os.path.exists('drivethru_model.zip'), '\u274c zip not created.'
print(f'\u2705 STEP 9 OK \u2014 drivethru_model.zip ({os.path.getsize("drivethru_model.zip")/1e6:.0f} MB) downloading...')
from google.colab import files
files.download('drivethru_model.zip')

## Step 10 — On your Mac: serve with Ollama (I'll walk you through this)
```bash
# install Ollama from https://ollama.com/download  (or: brew install ollama)
mkdir -p ~/drivethru_model && cd ~/drivethru_model
unzip ~/Downloads/drivethru_model.zip
ollama create drivethru -f Modelfile
ollama run drivethru 'lemme get a classic burger combo large'
```
When it replies with `<say>`+`<order>`, the model is ready. Tell Claude Code **"model's ready"** to start B2.

_Any red error in a cell?_ Paste it to Claude Code — Unsloth's API shifts and a cell occasionally needs a tweak.